In [1]:
import requests
import pandas as pd
from datetime import datetime, date, timezone, timedelta
import json
import time
import random
from typing import Any
import json
from pathlib import Path
import re

from renewables_permitting.utils import as_list, save_parquet, validate_required_columns

BASE_URL = "https://www.boe.es/datosabiertos/api/boe/sumario"

# PROJECT_ROOT = Path(__file__).resolve().parents[2]  # fuera del notebook
PROJECT_ROOT = Path.cwd().parent  # dentro del notebook

DATA_DIR = PROJECT_ROOT / "data"

BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"

BOE_DOCS_XML_DIR = BRONZE_DIR / "boe_docs_xml"

In [3]:
REQUIRED_XML_DOWNLOAD_COLS = {"identificador", "doc_file_stem", "url_xml"}

# Funciones

In [4]:
def build_xml_path(output_dir: Path, doc_file_stem: str) -> Path:
    return output_dir / f"{doc_file_stem}.xml"

In [5]:
def download_single_xml(
    identificador: str,
    doc_file_stem: str,
    url_xml: str,
    output_dir: Path,
    timeout: int = 30,
) -> dict:
    xml_path = build_xml_path(output_dir, doc_file_stem)

    record = {
        "identificador": identificador,
        "doc_file_stem": doc_file_stem,
        "url_xml": url_xml,
        "xml_path": str(xml_path),
        "download_status": None,
        "http_status_code": None,
        "downloaded_at": datetime.now(timezone.utc).isoformat(),
        "error_message": None,
        "file_size_bytes": None,
    }

    if pd.isna(url_xml) or not str(url_xml).strip():
        record["download_status"] = "missing_url"
        record["error_message"] = "missing url_xml"
        return record

    if xml_path.exists():
        record["download_status"] = "already_exists"
        record["file_size_bytes"] = xml_path.stat().st_size
        return record

    try:
        response = requests.get(url_xml, timeout=timeout)
        record["http_status_code"] = response.status_code

        if response.status_code != 200:
            record["download_status"] = "http_error"
            record["error_message"] = f"HTTP {response.status_code}"
            return record

        xml_path.write_bytes(response.content)

        record["download_status"] = "downloaded"
        record["file_size_bytes"] = xml_path.stat().st_size
        return record

    except requests.RequestException as exc:
        record["download_status"] = "request_error"
        record["error_message"] = str(exc)
        return record

In [6]:
def download_boe_xml_documents(
    candidates_path: Path,
    output_dir: Path,
    log_path: Path | None = None,
    limit: int | None = None,
    timeout: int = 30,
) -> pd.DataFrame:
    output_dir.mkdir(parents=True, exist_ok=True)

    if log_path is None:
        log_path = output_dir / "download_log.parquet"

    boe_candidates = pd.read_parquet(candidates_path)

    if limit is not None:
        boe_candidates = boe_candidates.head(limit).copy()

    validate_required_columns(
        boe_candidates,
        REQUIRED_XML_DOWNLOAD_COLS,
    )

    download_records = []

    for row in boe_candidates.itertuples(index=False):
        record = download_single_xml(
            identificador=row.identificador,
            doc_file_stem=row.doc_file_stem,
            url_xml=row.url_xml,
            output_dir=output_dir,
            timeout=timeout,
        )

        download_records.append(record)

    new_log = pd.DataFrame(download_records)

    if log_path.exists():
        old_log = pd.read_parquet(log_path)
        full_log = pd.concat([old_log, new_log], ignore_index=True)
    else:
        full_log = new_log

    full_log.to_parquet(log_path, index=False)

    return new_log

# Prueba

In [7]:
download_log = download_boe_xml_documents(
    candidates_path=SILVER_DIR / "boe_candidates" / "boe_candidates_normalized.parquet",
    output_dir=BOE_DOCS_XML_DIR
    #, limit=10,
)

In [8]:
df = pd.read_parquet(BOE_DOCS_XML_DIR / "download_log.parquet")
df

,identificador,doc_file_stem,url_xml,xml_path,download_status,http_status_code,downloaded_at,error_message,file_size_bytes
0,BOE-A-2023-10297,20230428_BOE-A-2023-10297,https://www.boe.es/diario_boe/xml.php?id=BOE-A...,/home/bgonzale/CiDaeN/_15_TrabajoFinMaster/tfm...,downloaded,200.0,2026-06-12T09:42:27.049858+00:00,None,37107
1,BOE-A-2023-10298,20230428_BOE-A-2023-10298,https://www.boe.es/diario_boe/xml.php?id=BOE-A...,/home/bgonzale/CiDaeN/_15_TrabajoFinMaster/tfm...,downloaded,200.0,2026-06-12T09:42:27.344572+00:00,None,29343
2,BOE-A-2023-10299,20230428_BOE-A-2023-10299,https://www.boe.es/diario_boe/xml.php?id=BOE-A...,/home/bgonzale/CiDaeN/_15_TrabajoFinMaster/tfm...,downloaded,200.0,2026-06-12T09:42:27.590600+00:00,None,34146
3,BOE-A-2023-10300,20230428_BOE-A-2023-10300,https://www.boe.es/diario_boe/xml.php?id=BOE-A...,/home/bgonzale/CiDaeN/_15_TrabajoFinMaster/tfm...,downloaded,200.0,2026-06-12T09:42:27.837012+00:00,None,26967
4,BOE-A-2023-10301,20230428_BOE-A-2023-10301,https://www.boe.es/diario_boe/xml.php?id=BOE-A...,/home/bgonzale/CiDaeN/_15_TrabajoFinMaster/tfm...,downloaded,200.0,2026-06-12T09:42:28.128034+00:00,None,29551
...,...,...,...,...,...,...,...,...,...
4818,BOE-A-2026-13451,20260620_BOE-A-2026-13451,https://www.boe.es/diario_boe/xml.php?id=BOE-A...,/home/bgonzale/CiDaeN/15_TrabajoFinMaster/tfm-...,already_exists,NaN,2026-06-23T20:56:13.105526+00:00,None,27848
4819,BOE-A-2026-13452,20260620_BOE-A-2026-13452,https://www.boe.es/diario_boe/xml.php?id=BOE-A...,/home/bgonzale/CiDaeN/15_TrabajoFinMaster/tfm-...,already_exists,NaN,2026-06-23T20:56:13.105540+00:00,None,27889
4820,BOE-A-2026-13453,20260620_BOE-A-2026-13453,https://www.boe.es/diario_boe/xml.php?id=BOE-A...,/home/bgonzale/CiDaeN/15_TrabajoFinMaster/tfm-...,already_exists,NaN,2026-06-23T20:56:13.105556+00:00,None,17383
4821,BOE-A-2026-13454,20260620_BOE-A-2026-13454,https://www.boe.es/diario_boe/xml.php?id=BOE-A...,/home/bgonzale/CiDaeN/15_TrabajoFinMaster/tfm-...,already_exists,NaN,2026-06-23T20:56:13.105574+00:00,None,39571


In [9]:
# TODO: Ahora mismo se ejecutan varias veces los archivos y en download_status aparece already_exists. Los archivos descargados correctamente no deberían ser reprocesados.

df.loc[df["identificador"] == "BOE-B-2026-19619"]

,identificador,doc_file_stem,url_xml,xml_path,download_status,http_status_code,downloaded_at,error_message,file_size_bytes
1156,BOE-B-2026-19619,20260611_BOE-B-2026-19619,https://www.boe.es/diario_boe/xml.php?id=BOE-B...,/home/bgonzale/CiDaeN/15_TrabajoFinMaster/tfm-...,downloaded,200.0,2026-06-19T15:13:19.305708+00:00,None,4145
2287,BOE-B-2026-19619,20260611_BOE-B-2026-19619,https://www.boe.es/diario_boe/xml.php?id=BOE-B...,/home/bgonzale/CiDaeN/15_TrabajoFinMaster/tfm-...,already_exists,NaN,2026-06-23T20:10:03.659394+00:00,None,4145
3544,BOE-B-2026-19619,20260611_BOE-B-2026-19619,https://www.boe.es/diario_boe/xml.php?id=BOE-B...,/home/bgonzale/CiDaeN/15_TrabajoFinMaster/tfm-...,already_exists,NaN,2026-06-23T20:21:29.923633+00:00,None,4145
4810,BOE-B-2026-19619,20260611_BOE-B-2026-19619,https://www.boe.es/diario_boe/xml.php?id=BOE-B...,/home/bgonzale/CiDaeN/15_TrabajoFinMaster/tfm-...,already_exists,NaN,2026-06-23T20:56:13.105409+00:00,None,4145
